# Scientific Research Agent

An AI agent that helps with literature questions. It downloads papers from arXiv, builds a searchable index
(text, tables, figure captions) and answers your questions **with cited evidence** such as `[2401.13744v3 p.6]`.
If the papers do not contain the answer, it says `INSUFFICIENT EVIDENCE` instead of guessing.

**How to use this notebook**

* **Part 1 (steps 1-5)** is all you need to chat with the agent. Run the cells from top to bottom.
  The quickest way is *Runtime → Run all*; the last cell of Part 1 is the chat and waits for your question.
* **Part 2 (steps 6-8)** checks how well the system works (retrieval scores, agent behaviour, an LLM judge).
  It is optional if you only want to ask questions.

**Before you start:** you need a free Gemini API key (https://aistudio.google.com/apikey).
In Colab, click the key icon in the left sidebar, add a secret named `GEMINI_API_KEY`, paste the key,
and switch on *Notebook access*. The key stays in Colab Secrets and is never shown in this notebook.

If you restart the Colab session, run steps 1 and 2 again. Downloaded papers stay, the settings do not.

## Part 1: Ask the agent

### Step 1. Get the code and install the packages
This takes about a minute. It copies the code from GitHub into Colab and installs the Python packages it needs.

In [3]:
import os

# Address of the repository with the code, and the folder to put it in
REPO_URL = "https://github.com/Yul04ek/scientific-research-agent.git"
FOLDER = "/content/research-agent"

# Download the code only if it is not there yet
if not os.path.exists(FOLDER):
    !git clone {REPO_URL} {FOLDER}
%cd {FOLDER}

# Install the Python packages listed in the requirements file
!pip install -q -r requirements-research.txt

/content/research-agent


### Step 2. Connect the model
The agent uses Google's Gemini model. This cell reads your key from Colab Secrets, sets the options below,
and sends one short test message. The test should print `ready`.

| Setting | Meaning |
|---|---|
| `MODEL_NAME` | Gemini model. The free tier allows about 500 requests per day. If the model is overloaded (error 503), try `gemini-3.5-flash-lite`. |
| `EMBED_BACKEND` | `local` means the search index is computed in Colab and uses no API quota. |
| `EVIDENCE_THRESHOLD` | Minimum similarity between the question and the best passage. Below it, the agent warns that evidence is weak. It is not set here: step 4 calculates it for your papers. |
| `MAX_MODEL_REQUESTS` | Upper limit of model calls for one question, so one question cannot use up the daily quota. |

In [4]:
import os
from google.colab import userdata

# Read the key from Colab Secrets
try:
    os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError(
        "Secret GEMINI_API_KEY not found. Add it with the key icon in the left sidebar, "
        "switch on 'Notebook access', and run this cell again."
    )

os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini through the OpenAI-compatible API
os.environ["MODEL_NAME"] = "gemini-3.5-flash-lite"
os.environ["EMBED_BACKEND"] = "local"
os.environ["DATA_DIR"] = "data"               # folder for PDFs and the search index
os.environ["MAX_MODEL_REQUESTS"] = "15"
os.environ["PYDANTIC_AI_NO_BANNER"] = "1"     # hide a start banner in the output

# Send one short message to check that the key and the model work (30 second limit)
from openai import OpenAI

client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
reply = client.chat.completions.create(
    model=os.environ["MODEL_NAME"],
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
)
print(reply.choices[0].message.content)

ready


### Step 3. Choose a topic and collect papers
Run the next cell and type the topic you want to explore (plain words are fine, for example `integral equations`).
The cell first shows the 5 best matches on arXiv and asks you to confirm, so a mistyped topic costs one search
and not a full download. Then the agent downloads open-access papers from arXiv, splits them into passages,
and builds the search index. This takes a few minutes.

* The arXiv server is sometimes busy (error 503). The cell then waits and tries again by itself.
* If the corpus already has papers, the cell asks whether to keep them. Type `new` to delete them and start a new topic.
* Step 4 then calibrates the evidence threshold for the new papers.

In [5]:
import shutil
import time
from coding_assistant.research import corpus
from coding_assistant.capabilities.literature_search import search_arxiv, collect_papers

N_PAPERS = 12   # number of papers to download (1 to 15)


def with_retry(function, *args, tries=4, wait=30):
    # arXiv is sometimes busy (error 503): wait a little and try the same call again
    for attempt in range(tries):
        result = function(*args)
        if not result.startswith("ERROR"):
            return result
        print(result, f"Waiting {wait} s, then trying again ({attempt + 1}/{tries}) ...")
        time.sleep(wait)
    return result


papers_in_corpus = corpus.load_meta()
if papers_in_corpus:
    print(f"The corpus already has {len(papers_in_corpus)} papers.")
    answer = input("Press Enter to keep them, or type 'new' to delete them and start a new topic: ")
    if answer.strip().lower() == "new":
        shutil.rmtree("data", ignore_errors=True)   # remove the old PDFs and the old index
        papers_in_corpus = {}

if not papers_in_corpus:
    while True:
        topic = input("Which topic do you want to explore? ").strip()
        if not topic:
            print("Please type a topic, for example: integral equations")
            continue
        # Plain words become a phrase search; text that already uses arXiv syntax (for example all:...) is used as it is
        query = topic if ":" in topic else f'all:"{topic}"'
        # Show the best matches first: a wrong topic then costs one search, not a full download
        print(with_retry(search_arxiv, query, 5))
        answer = input("Download papers for this topic? Press Enter for yes, or type 'n' to choose another topic: ")
        if answer.strip().lower() != "n":
            break

    print(with_retry(collect_papers, query, N_PAPERS))
    n_papers = len(corpus.load_meta())
    if n_papers:
        print(f"The corpus now has {n_papers} papers. Run step 4 next.")
    else:
        print("No papers were collected (nothing found, or arXiv did not answer). Wait a few minutes and run this cell again.")

Which topic do you want to explore? boundary element method
1509.00440v1 | 2015 | math.NA | A non-symmetric coupling of the finite volume method and the boundary element method
1912.05789v2 | 2019 | math.NA | Functional a posteriori error estimates for boundary element methods
2502.21008v1 | 2025 | math.NA | A spectral boundary element method for acoustic interference problems
2206.00708v3 | 2022 | math.NA | The Boundary Element Method for Acoustic Transmission with Nonconforming Grids
1406.2817v1 | 2014 | math.NA | Isogeometric Boundary Element Method with Hierarchical Matrices
Download papers for this topic? Press Enter for yes, or type 'n' to choose another topic: 


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Added 11 papers; corpus now has 12 papers and 730 chunks.
+ 1509.00440v1 A non-symmetric coupling of the finite volume method and the boundary element me
+ 1912.05789v2 Functional a posteriori error estimates for boundary element methods
+ 2502.21008v1 A spectral boundary element method for acoustic interference problems
+ 2206.00708v3 The Boundary Element Method for Acoustic Transmission with Nonconforming Grids
+ 1406.2817v1 Isogeometric Boundary Element Method with Hierarchical Matrices
+ physics/0206018v2 Boundary element method for resonances in dielectric microcavities
+ 1808.06589v1 Modelling a conductive-capacitive medium using the boundary element method
+ 2204.09549v1 Isogeometric boundary element method for acoustic scattering by a submarine
+ 1704.07745v4 A hybrid numerical-asymptotic boundary element method for high frequency scatter
+ 1302.6054v1 Quadrature for second-order triangles in the Boundary Element Method
+ 1702.04224v1 Local convergence of the boundary element m

**Topic map (the machine-learning part).** The agent groups the papers by topic. It turns each abstract into a vector,
clusters the vectors with KMeans (the number of clusters is chosen by the silhouette score), and checks the result
against the arXiv categories of the papers, which were not used for the clustering. With few papers the clusters are weak,
so read this as a rough overview, not as a result.

In [6]:
from coding_assistant.capabilities.corpus_analysis import corpus_overview

print(corpus_overview())

12 papers grouped into 2 topic clusters (silhouette 0.144).
Check against arXiv categories ['math.NA', 'physics.app-ph', 'physics.optics']: NMI 0.21, ARI 0.032 (1 = perfect agreement, 0 = no better than chance).
- Cluster 0 (6 papers), key terms: error, method, local, element, boundary, transformation. E.g. 1912.05789v2 Functional a posteriori error estimates for boundary element methods; 1702.04224v1 Local convergence of the boundary element method on polyhedral domains; 1302.6054v1 Quadrature for second-order triangles in the Boundary Element Method
- Cluster 1 (6 papers), key terms: acoustic, boundary, method, element, boundary element, scattering. E.g. 1704.07745v4 A hybrid numerical-asymptotic boundary element method for high frequen; 2502.21008v1 A spectral boundary element method for acoustic interference problems; 2206.00708v3 The Boundary Element Method for Acoustic Transmission with Nonconformi


### Step 4. Calibrate the evidence threshold
Run this cell every time. When no passage is close enough to the question, the agent should say `INSUFFICIENT EVIDENCE`.
"Close enough" is a number, the **threshold**, and a good value depends on the papers. The cell:

1. uses the saved threshold if this corpus was calibrated before, and stops there;
2. otherwise lets the model write test questions from your papers (about 20 model calls, a few minutes),
   runs the search on them and on 8 off-topic questions that the papers cannot answer,
   and chooses the threshold that separates the two groups best;
3. saves the value in `data/threshold.txt`, next to the papers it belongs to.

The 8 off-topic questions are fixed (biology, economics, history, ...). If your topic is close to one of them,
edit `eval/unanswerable.jsonl` first. With few papers the value is rough. The report in step 6 shows how well it separates the groups.
If the agent later downloads more papers during the chat, the saved threshold stays as it was.

In [7]:
import json
import os

threshold_file = "data/threshold.txt"   # saved next to the corpus, so it belongs to these papers

if os.path.exists(threshold_file):
    os.environ["EVIDENCE_THRESHOLD"] = open(threshold_file).read().strip()
    print("Using the saved threshold for this corpus:", os.environ["EVIDENCE_THRESHOLD"])
else:
    os.environ["N_EVAL_QUESTIONS"] = "20"   # number of test questions the model writes

    # Write test questions from the corpus, then measure the search quality and the threshold
    !python -m eval.build_eval_set
    !python -m eval.run_eval retrieval

    # Read the suggested threshold from the saved result, use it, and save it
    calibration = json.load(open("eval/results/retrieval.json"))["calibration"]
    if calibration:
        os.environ["EVIDENCE_THRESHOLD"] = str(calibration["threshold"])
        open(threshold_file, "w").write(str(calibration["threshold"]))
        print("Evidence threshold set to", calibration["threshold"])
    else:
        print("Calibration was not possible. The default threshold stays:", os.environ.get("EVIDENCE_THRESHOLD", "0.45"))

Model answered 429, waiting 20 s ...
Model answered 429, waiting 40 s ...
Wrote 20 questions to eval/retrieval_set.jsonl
Loading weights: 100% 199/199 [00:00<00:00, 4330.26it/s]
## Retrieval quality
20 synthetic questions, 730 chunks, k=6, max 2 chunks per paper.

| mode | level | hit@1 | hit@3 | hit@6 | MRR |
|---|---|---|---|---|---|
| dense | paper | 0.85 | 0.95 | 1.0 | 0.9 |
| dense | page | 0.55 | 0.8 | 0.8 | 0.658 |
| bm25 | paper | 0.75 | 0.95 | 0.95 | 0.842 |
| bm25 | page | 0.55 | 0.75 | 0.75 | 0.642 |
| hybrid | paper | 0.8 | 1.0 | 1.0 | 0.883 |
| hybrid | page | 0.65 | 0.9 | 0.9 | 0.758 |

## Weak-evidence threshold
Best dense similarity: answerable median 0.82, unanswerable median 0.58 (8 off-topic questions).
AUROC 1.0; best threshold **0.744** (balanced accuracy 1.0). Set it with `EVIDENCE_THRESHOLD=0.744`.
Evidence threshold set to 0.744


### Step 5. Chat with the agent
Run the three cells below, in this order: a safety setting, a list of example questions, and the chat itself.

**Safety setting.** The agent may decide to download more papers, for example when a question is outside your topic.
With this setting it first asks you `Allow? [y/N]`. Type `y` (English keyboard layout) to allow it or `n` to refuse.
Nothing is downloaded without your answer.

**How to chat**

* Type your question and press **Enter twice**: once after typing, and once more in the empty field that appears.
  Nothing is sent before the second Enter. A question can have several lines.
* The agent chooses its tools itself (literature search, retrieval, figure inspection, Python) and prints each tool call.
* Add `@low` or `@high` to a question to set the reasoning effort (`@low` is faster, `@high` thinks longer).
* The chat remembers the earlier questions of this session.
* For calculations the agent writes a Python script into the folder `sandbox/` (`run_001.py`, `run_002.py`, ...). Open a file in the file browser to check what was computed. The files stay until the Colab session ends, so download them if you need them later.
* To end the chat, type `exit` and press Enter. A cell that waits for input cannot be stopped with the stop button.
  If the notebook gets stuck: *Runtime → Restart session*, then *Runtime → Run all*, and press Enter once at step 3.

Known limits: tables and figure captions are extracted imperfectly (see the README), and the free Gemini tier
is sometimes overloaded. In that case the agent waits and tries again by itself.

In [8]:
import os

# Ask the person before the agent downloads papers
os.environ["CONFIRM_COLLECT"] = "1"

In [9]:
import json
import os

# Show three questions that were written from your papers in step 4
path = "eval/retrieval_set.jsonl"
if os.path.exists(path):
    for line in open(path).read().splitlines()[:3]:
        print("-", json.loads(line)["question"])
else:
    print("Run step 4 first to get example questions from your papers.")

# Two questions that work for any topic
print("- What sub-topics does this literature contain?   (overview)")
print("- What does the corpus say about CRISPR off-target effects in zebrafish?   (off topic: expect INSUFFICIENT EVIDENCE; if the agent asks to download papers, answer n)")

- What is the value of the normal derivative integral of the fundamental solution over a boundary $\Gamma$ when the source point is located at an edge or a vertex of a cubic domain $\Omega^-$?
- How does the relative $L^2(\Gamma)$ error vary with the number of elements per wavelength for different polynomial orders in the interior acoustic problem of a torus?
- How is the error bound for the bilinear form difference $A - \mathcal{A}_V$ bounded in terms of the model data and shape regularity constants during the proof of convergence for the finite element discretization?
- What sub-topics does this literature contain?   (overview)
- What does the corpus say about CRISPR off-target effects in zebrafish?   (off topic: expect INSUFFICIENT EVIDENCE; if the agent asks to download papers, answer n)


In [10]:
from coding_assistant.research_agent import run_agent

# Start the chat loop: read a question, run the agent, print the answer, repeat (type exit to stop)
await run_agent()

Research assistant ready. Type a question and press Enter twice to send it. Type exit to stop. Tip: @low / @high 
set reasoning effort.

>>

What is the value of the normal derivative integral of the fundamental solution over a boundary $\Gamma$ when the source point is located at an edge or a vertex of a cubic domain $\Omega^-$?


...

[11:15:16] Calling tool: search_arxiv(query='"normal derivative" "fundamental solution" "edge"   ]8;id=708889;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=544898;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\
           "vertex" "', max_results='10', sort_by='relevance')                                                     

...

[11:40:08] Calling tool: load_skill(skill_name='conflicting-evidence')                           ]8;id=861366;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=213961;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\

[11:40:09] Calling tool: load_skill(skill_name='scientific-qa')                                  ]8;id=752804;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=832055;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\

INSUFFICIENT EVIDENCE: The corpus does not contain papers on the boundary element method for Kirchhoff thin plates 
(the collected papers focus on acoustic scattering, elastostatics with Lamé-Navier equations, and potential        
problems). Furthermore, the tool execution prevented collecting new papers on this specific topic.

>>

exit


## Part 2: Check how well it works

These steps evaluate the system. They are optional for everyday use, and they do **not** run automatically
(also not with *Run all*): the cells below only do their work when `RUN_EVALUATION = True`.

In [16]:
# Set to True to run the evaluation. It uses many model calls from the free daily quota.
RUN_EVALUATION = True

### Step 6. Retrieval quality and the threshold
Step 4 already measured retrieval. This cell prints the saved report.

* **hit@k**: share of test questions for which the source passage is among the first k results (k = 1, 3, 6).
* **MRR**: average of 1 divided by the rank of the source passage (1 is best).
* The test questions are written by a model from the passages themselves, so they share words with the answer.
  Real questions are harder, and these scores are optimistic.

In [17]:
from pathlib import Path

report_file = Path("eval/results/retrieval.md")
print(report_file.read_text() if report_file.exists() else "No retrieval report yet. Run step 4 first.")

## Retrieval quality
20 synthetic questions, 730 chunks, k=6, max 2 chunks per paper.

| mode | level | hit@1 | hit@3 | hit@6 | MRR |
|---|---|---|---|---|---|
| dense | paper | 0.85 | 0.95 | 1.0 | 0.9 |
| dense | page | 0.55 | 0.8 | 0.8 | 0.658 |
| bm25 | paper | 0.75 | 0.95 | 0.95 | 0.842 |
| bm25 | page | 0.55 | 0.75 | 0.75 | 0.642 |
| hybrid | paper | 0.8 | 1.0 | 1.0 | 0.883 |
| hybrid | page | 0.65 | 0.9 | 0.9 | 0.758 |

## Weak-evidence threshold
Best dense similarity: answerable median 0.82, unanswerable median 0.58 (8 off-topic questions).
AUROC 1.0; best threshold **0.744** (balanced accuracy 1.0). Set it with `EVIDENCE_THRESHOLD=0.744`.


### Step 7. Agent evaluation with an LLM judge
Set `RUN_EVALUATION = True` in the cell above first. The agent answers the questions in `eval/agent_questions.jsonl`.
The questions are written for the **default topic** (conformal prediction). For another topic, edit that file first
(in the Colab file browser: `research-agent/eval/agent_questions.jsonl`).

Automatic checks, per question:

* the expected tools were used (for example `run_python` for a computation, `inspect_figure` for a figure);
* every citation like `[paper_id p.N]` was really returned by a tool during that run (this detects invented citations,
  but it does not check that the cited passage supports the claim);
* on questions the papers cannot answer, the answer says `INSUFFICIENT EVIDENCE`; on answerable questions it does not;
* an expected keyword appears in the answer, where one is defined.

An **LLM judge** (the same model) then scores from 1 to 5 whether every claim in the answer is supported by the
retrieved evidence. The judge is a weak signal: it is the same kind of model that wrote the answer.

The papers are fixed during the run (the agent may not collect new ones). A question that fails with a server error
is recorded and the run continues. Expect 10 to 20 minutes: each question may use up to 15 model calls, and Gemini
can be slow.

In [18]:
import json

# Show the questions that will be asked
for line in open("eval/agent_questions.jsonl"):
    item = json.loads(line)
    print(item["id"], "|", item["type"], "|", item["question"][:100])

q1 | answerable | What value does the integral of the normal derivative of the fundamental solution over the boundary 
q2 | answerable | What convergence rate is proved for the Galerkin boundary element method applied to the hyper-singul
q3 | compute | Check numerically: for the unit circle in 2D with the fundamental solution -(1/(2*pi))*ln|x-y|, comp
q4 | overview | What sub-topics or schools of thought does this literature contain?
q5 | figure | Find a figure or table that shows an error against the number of elements (for example per wavelengt
q6 | unanswerable | What does the corpus say about CRISPR off-target effects in zebrafish?
q7 | unanswerable | Which paper in the corpus reports results on the MNIST-9000 benchmark for quantum annealers?


In [19]:
if RUN_EVALUATION:
    from eval import run_eval

    # Run all questions, with the LLM judge. The report is also saved in eval/results/.
    # To repeat only some questions:  await run_eval.run_agent_eval(True, only=["q6", "q7"])
    report = await run_eval.run_agent_eval(use_judge=True)
    print(report)
else:
    print("Skipped. Set RUN_EVALUATION = True above to run the agent evaluation.")

[12:25:02] Calling tool: load_skill(skill_name='scientific-qa')                                  ]8;id=685161;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=147348;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\

           Calling tool: list_papers()                                                           ]8;id=532165;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=25738;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\

[12:25:03] Calling tool: retrieve_evidence(query='normal derivative fundamental solution         ]8;id=967612;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=700700;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\
           integral boundary dom', k='6', paper_id='None')                                                         

[12:25:05] Calling tool: run_python(code="# Let's verify standard notation and values from the   ]8;id=715886;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=818694;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\
           text in", timeout_seconds='90')                                                                         

q1 answerable   passed=True tools=['list_papers', 'load_skill', 'retrieve_evidence', 'run_python']


...

           Calling tool: search_arxiv(query='MNIST-9000', max_results='10', sort_by='relevance') ]8;id=589912;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=296280;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\

[12:29:09] Calling tool: search_arxiv(query='MNIST 9000 quantum annealer OR "quantum annealing"  ]8;id=210020;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=708569;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\
           OR D-Wave', max_results='10', sort_by='relevance')                                                      

[12:29:13] Calling tool: search_arxiv(query='"MNIST-9000" OR "MNIST 9000"', max_results='10',    ]8;id=838220;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=331909;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\
           sort_by='relevance')                                                                                    

[12:29:17] Calling tool: search_arxiv(query='MNIST quantum annealer OR "quantum annealing"',     ]8;id=561532;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py\tool_logging.py]8;;\:]8;id=405014;file:///content/research-agent/coding_assistant/capabilities/tool_logging.py#25\25]8;;\
           max_results='10', sort_by='relevance')                                                                  

q7 unanswerable passed=False tools=['collect_papers', 'list_papers', 'search_arxiv']
## Agent evaluation
6/7 questions passed all checks.

| id | type | passed | failed checks | tools used |
|---|---|---|---|---|
| q1 | answerable | yes | - | list_papers, load_skill, retrieve_evidence, run_python |
| q2 | answerable | yes | - | collect_papers, list_papers, retrieve_evidence, search_arxiv |
| q3 | compute | yes | - | list_papers, load_skill, retrieve_evidence, run_python |
| q4 | overview | yes | - | corpus_overview, list_papers |
| q5 | figure | yes | - | inspect_figure, list_figures, list_papers, retrieve_evidence, search_arxiv |
| q6 | unanswerable | yes | - | list_papers, load_skill |
| q7 | unanswerable | NO | abstains | collect_papers, list_papers, search_arxiv |

LLM-judge faithfulness (1-5): mean 4.50 over 4 answers.
Citations: 14/27 refer to passages the tools really returned.


### Step 8. Save the results
Everything in Colab is deleted when the session ends. This cell packs the evaluation results and downloads them
as a zip file (only when `RUN_EVALUATION = True`). The papers and the index are in the folder `data/`.

In [20]:
if RUN_EVALUATION:
    # Pack the results and the generated test questions, then download the zip
    !zip -qr /content/results.zip eval/results eval/retrieval_set.jsonl

    from google.colab import files
    files.download("/content/results.zip")
else:
    print("Skipped. Set RUN_EVALUATION = True above to save the evaluation results.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>